In [ ]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv('dataset.csv')

In [4]:
df.head()  #1 means real news 0 means fake news

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [5]:
df.shape

(45757, 2)

In [6]:
df['text'][1]

'German parties start to find common ground in coalition talks BERLIN (Reuters) - Parties seeking to form Germany s next government found common ground in areas of social policy and digital infrastructure on Monday, but remain far apart on issues that provoked stormy clashes last week. Talks between conservatives, Greens and Free Democrats (FDP) resumed on Monday after Chancellor Angela Merkel convened a weekend meeting to clear the air between ideologically diverse parties forced by electoral arithmetic into awkward partnership.  The weekend pause for thought did us good,  said Andreas Scheuer, a leader in the conservative Christian Social Union (CSU), the Bavarian sister party to Merkel s Christian Democrats (CDU). Other party leaders agreed. But Monday s talks, on education, digitalization, pensions and labor issues, as well as domestic security, were always expected to be less contentious than the immigration, fiscal and climate policies that divided them last week. One sign of the

In [7]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [8]:
df['text'].duplicated().sum()

0

In [ ]:
df['label'].value_counts() 

label
1    22900
0    22857
Name: count, dtype: int64

In [10]:
df.head(15)

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0
5,President Curtsy Delivers A Really Sick Burn ...,0
6,"NSA risks talent exodus amid morale slump, Tru...",1
7,Trump Completely COLLAPSING In Most Important...,0
8,John Kerry commits more U.S. military aid for ...,1
9,BOOM! Hey Democrats….Why The Violent Riots? YO...,0


# preprocessing

In [11]:
df['text'] = df['text'].str.lower()

In [12]:
df.head()

,text,label
0,gere faults trump for blurring meaning of 'ref...,1
1,german parties start to find common ground in ...,1
2,senate democratic leader says attorney general...,1
3,"tennis: kyrgios fined $10,000 for shanghai wal...",1
4,trump threw mar-a-lago fundraiser for woman a...,0


In [13]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join(
        word for word in text.split()
        if word.lower() not in stop_words
    )


In [14]:
df['text'] = df['text'].apply(remove_stopwords)

In [15]:
df.head()

,text,label
0,gere faults trump blurring meaning 'refugee' '...,1
1,german parties start find common ground coalit...,1
2,senate democratic leader says attorney general...,1
3,"tennis: kyrgios fined $10,000 shanghai walk te...",1
4,trump threw mar-a-lago fundraiser woman center...,0


In [16]:
df['text'][1]

'german parties start find common ground coalition talks berlin (reuters) - parties seeking form germany next government found common ground areas social policy digital infrastructure monday, remain far apart issues provoked stormy clashes last week. talks conservatives, greens free democrats (fdp) resumed monday chancellor angela merkel convened weekend meeting clear air ideologically diverse parties forced electoral arithmetic awkward partnership. weekend pause thought us good, said andreas scheuer, leader conservative christian social union (csu), bavarian sister party merkel christian democrats (cdu). party leaders agreed. monday talks, education, digitalization, pensions labor issues, well domestic security, always expected less contentious immigration, fiscal climate policies divided last week. one sign division came interview merkel ally peter altmaier gave newspaper die zeit, shot media reports reshuffling ministerial portfolios. der spiegel magazine reported fdp might head wea

In [17]:
import string

df['punct_count'] = df['text'].apply(
    lambda x: sum(1 for char in x if char in string.punctuation)
)

df.groupby('label')['punct_count'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,22857.0,51.609398,79.653637,0.0,16.0,44.0,66.0,7295.0
1,22900.0,55.443930,44.897107,1.0,23.0,47.0,73.0,1144.0


In [18]:
df[df['label']==0].sort_values('punct_count', ascending=False)[['text','punct_count']].head(10)

,text,punct_count
32793,fed republicans threaten third party option tr...,7295
36842,stephen colbert gets donald rumsfeld admit ira...,4836
32225,god bible architect great pyramid - unlocking ...,1564
26592,state’s economy rank? washington dc’s ranking ...,943
16167,obama commutes 61 prisoners’ sentences…here’s ...,920
43443,wow! america attack 187 organizations directly...,789
21642,anti-american george soros locks arms nfl trum...,765
16923,durham bulls 2017 - part 2 - pitchers durham b...,754
20,we’ve got list bill hillary clinton’s speeches...,724
17544,fundamental constitutional rights negated real...,714


In [19]:
df['text'][32793]

"fed republicans threaten third party option trump nominee saying long time republican party verge splitting right. sort of. always assumed establishment would control party iron fist tea party white supremacists would abandon gop form white power party. rise donald trump, looks like establishment going abandon party run candidate raving sociopath:spurred donald j. trump mounting victories, small influential growing group conservative leaders calling third-party option spare voters wrenching general election choice republican consider completely unacceptable hillary clinton.while gained intense popularity right, mr. trump alienated key blocs republican coalition slash-and-burn campaign. many, initial refusal last weekend disavow endorsement david duke, white supremacist, breaking point. breaking point ass. problem conservative leaders trump racism overt nature it. republicans relied white resentment 50 years win elections, keep concealed lest get publicly torn apart party old white rac

In [20]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    return text


In [21]:
df['text'] = df['text'].apply(clean_text)

In [22]:
df['punct_count'] = df['text'].apply(
    lambda x: sum(1 for c in x if c in string.punctuation)
)

In [23]:
df.groupby('label')['punct_count'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,22857.0,49.121582,53.097003,0.0,15.0,43.0,64.0,943.0
1,22900.0,55.412052,44.803486,1.0,23.0,47.0,73.0,1139.0


In [24]:
df['text'][32793]

'fed republicans threaten third party option trump nominee saying long time republican party verge splitting right. sort of. always assumed establishment would control party iron fist tea party white supremacists would abandon gop form white power party. rise donald trump, looks like establishment going abandon party run candidate raving sociopath:spurred donald j. trump mounting victories, small influential growing group conservative leaders calling third-party option spare voters wrenching general election choice republican consider completely unacceptable hillary clinton.while gained intense popularity right, mr. trump alienated key blocs republican coalition slash-and-burn campaign. many, initial refusal last weekend disavow endorsement david duke, white supremacist, breaking point. breaking point ass. problem conservative leaders trump racism overt nature it. republicans relied white resentment 50 years win elections, keep concealed lest get publicly torn apart party old white rac

In [25]:
df[df['punct_count'] > 1000][['label', 'text']]

,label,text
3568,1,portraits las vegas shooting victims (cnn) one...


In [26]:
import re

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

df['text'] = df['text'].apply(remove_punctuation)

In [27]:
from collections import Counter
import pandas as pd

fake_top15 = Counter(
    ' '.join(df[df['label']==0]['text']).split()
).most_common(15)

real_top15 = Counter(
    ' '.join(df[df['label']==1]['text']).split()
).most_common(15)

top_words = pd.DataFrame({
    'Fake Word': [w for w,c in fake_top15],
    'Fake Count': [c for w,c in fake_top15],
    'Real Word': [w for w,c in real_top15],
    'Real Count': [c for w,c in real_top15]
})

print(top_words)

    Fake Word  Fake Count   Real Word  Real Count
0       trump       78259        said      105467
1        said       29333       trump       48973
2      people       29172          us       47017
3          us       26942       would       33755
4         one       25945     reuters       29055
5       would       25924   president       27174
6   president       23937       state       20047
7     clinton       21160         new       19884
8        like       19533  government       19073
9      donald       18265      states       18554
10        new       16760       house       18366
11       also       16668        also       17554
12      obama       16630      people       16789
13    hillary       16416      united       16319
14       even       15634  republican       16147


In [28]:
import re

def remove_special_chars(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

df['text'] = df['text'].apply(remove_special_chars)

In [29]:
df.head()

,text,label,punct_count
0,gere faults trump blurring meaning refugee ter...,1,58
1,german parties start find common ground coalit...,1,49
2,senate democratic leader says attorney general...,1,11
3,tennis kyrgios fined 10000 shanghai walk tenni...,1,43
4,trump threw maralago fundraiser woman center b...,0,51


In [30]:
df['text'][0]

'gere faults trump blurring meaning refugee terrorist berlin reuters  actor activist richard gere said friday us president donald trump managed merge meaning words refugee terrorist minds many americans gere also told news conference berlin film festival world premiere new film the dinner found discouraging see term refugee go dispiriting change meaning united states the horrible thing trump done conflated two words  refugee terrorist gere 67 told 100 journalists it means thing us now thats hes accomplished large segment population trump ordered travel ban refugees citizens seven muslimmajority countries jan 27 us appeal court san francisco refused reinstate temporary ban order trump criticized court decision a refugee used someone empathy someone wanted help wanted give refuge to gere said now were afraid is biggest crime itself conflating two ideas gere met chancellor angela merkel week berlin festival told trump phone call two weeks ago global fight terrorism excuse banning people m

In [31]:
import re

def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

df['text'] = df['text'].apply(remove_extra_spaces)

# vectorization and model training

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

tfidf = TfidfVectorizer()

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [33]:
X_test.shape

(9152, 230793)

In [34]:
X_train.shape

(36605, 230793)

In [35]:
len(tfidf.vocabulary_)

230793

In [36]:
X_train.nnz

6657481

In [37]:
from sklearn.naive_bayes import MultinomialNB

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [39]:
model1 = MultinomialNB()
model1.fit(X_train, y_train)

MultinomialNB()

In [40]:
y_pred1 = model1.predict(X_test)

In [41]:
print("Accuracy:", accuracy_score(y_test, y_pred1))
print("Precision:", precision_score(y_test, y_pred1))
print("Recall:", recall_score(y_test, y_pred1))
print("F1 Score:", f1_score(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))

Accuracy: 0.9207823426573427
Precision: 0.9134386496429344
Recall: 0.9285085789705235
F1 Score: 0.9209119668375695
[[4206  400]
 [ 325 4221]]


In [42]:
from sklearn.linear_model import LogisticRegression
model2=LogisticRegression()
model2.fit(X_train,y_train)

LogisticRegression()

In [43]:
y_pred2=model2.predict(X_test)

In [44]:
print("Accuracy:", accuracy_score(y_test, y_pred2))
print("Precision:", precision_score(y_test, y_pred2))
print("Recall:", recall_score(y_test, y_pred2))
print("F1 Score:", f1_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

Accuracy: 0.9765078671328671
Precision: 0.9791989378181013
Recall: 0.973383194016718
F1 Score: 0.9762824048538334
[[4512   94]
 [ 121 4425]]


In [45]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

model3=LinearSVC()
model4 = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)


In [46]:
model3.fit(X_train,y_train)

c:\Users\Shashwat\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC()

In [47]:
y_pred3=model3.predict(X_test)

In [63]:
print("Accuracy:", accuracy_score(y_test, y_pred3))
print("Precision:", precision_score(y_test, y_pred3))
print("Recall:", recall_score(y_test, y_pred3))
print("F1 Score:", f1_score(y_test, y_pred3))
print(confusion_matrix(y_test, y_pred3))

Accuracy: 0.9854676573426573
Precision: 0.9852650098966351
Recall: 0.9854817421909371
F1 Score: 0.9853733641262509
[[4539   67]
 [  66 4480]]


In [48]:
model4.fit(X_train,y_train)
y_pred4=model4.predict(X_test)

In [49]:
print("Accuracy:", accuracy_score(y_test, y_pred4))
print("Precision:", precision_score(y_test, y_pred4))
print("Recall:", recall_score(y_test, y_pred4))
print("F1 Score:", f1_score(y_test, y_pred4))
print(confusion_matrix(y_test, y_pred4))

Accuracy: 0.9497377622377622
Precision: 0.9519911504424778
Recall: 0.9465464144302683
F1 Score: 0.9492609750716965
[[4389  217]
 [ 243 4303]]


In [52]:
from xgboost import XGBClassifier
model5=XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

In [ ]:
model5.fit(X_train,y_train)
y_pred5=model5.predict(X_test)

In [54]:
print("Accuracy:", accuracy_score(y_test, y_pred5))
print("Precision:", precision_score(y_test, y_pred5))
print("Recall:", recall_score(y_test, y_pred5))
print("F1 Score:", f1_score(y_test, y_pred5))
print(confusion_matrix(y_test, y_pred5))

Accuracy: 0.9888548951048951
Precision: 0.9931202840656902
Recall: 0.984381874175099
F1 Score: 0.9887317719840919
[[4575   31]
 [  71 4475]]


In [55]:
print("Train Accuracy:", model5.score(X_train, y_train))
print("Test Accuracy:", model5.score(X_test, y_test))

Train Accuracy: 0.9974866821472477
Test Accuracy: 0.9888548951048951


# optimisation

will reduce the tfidf corpus now to see if we get any further optimisation

In [56]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

tfidf = TfidfVectorizer(
    max_features=20000,
    min_df=5,
    max_df=0.9
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [57]:
model1 = MultinomialNB()
model1.fit(X_train, y_train)
y_pred1=model1.predict(X_test)

In [58]:
print("Accuracy:", accuracy_score(y_test, y_pred1))
print("Precision:", precision_score(y_test, y_pred1))
print("Recall:", recall_score(y_test, y_pred1))
print("F1 Score:", f1_score(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))

Accuracy: 0.9211101398601399
Precision: 0.9277404921700224
Recall: 0.9122305323361196
F1 Score: 0.9199201419698314
[[4283  323]
 [ 399 4147]]


In [59]:
model2=LogisticRegression()
model2.fit(X_train,y_train)
y_pred2=model2.predict(X_test)

In [60]:
print("Accuracy:", accuracy_score(y_test, y_pred2))
print("Precision:", precision_score(y_test, y_pred2))
print("Recall:", recall_score(y_test, y_pred2))
print("F1 Score:", f1_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

Accuracy: 0.9779283216783217
Precision: 0.9794701986754967
Recall: 0.9760228772547295
F1 Score: 0.9777434993389158
[[4513   93]
 [ 109 4437]]


In [61]:
model3=LinearSVC()
model4 = RandomForestClassifier(
    n_estimators=50,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)
model3.fit(X_train,y_train)
y_pred3=model3.predict(X_test)

c:\Users\Shashwat\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


In [62]:
model4.fit(X_train,y_train)
y_pred4=model4.predict(X_test)

In [63]:
print("Accuracy:", accuracy_score(y_test, y_pred3))
print("Precision:", precision_score(y_test, y_pred3))
print("Recall:", recall_score(y_test, y_pred3))
print("F1 Score:", f1_score(y_test, y_pred3))
print(confusion_matrix(y_test, y_pred3))

Accuracy: 0.9856861888111889
Precision: 0.9852714882391734
Recall: 0.9859216893972723
F1 Score: 0.9855964815832875
[[4539   67]
 [  64 4482]]


In [64]:
print("Accuracy:", accuracy_score(y_test, y_pred4))
print("Precision:", precision_score(y_test, y_pred4))
print("Recall:", recall_score(y_test, y_pred4))
print("F1 Score:", f1_score(y_test, y_pred4))
print(confusion_matrix(y_test, y_pred4))

Accuracy: 0.9621940559440559
Precision: 0.9658385093167702
Recall: 0.957765068191817
F1 Score: 0.9617848464766954
[[4452  154]
 [ 192 4354]]


In [65]:
model5=XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
model5.fit(X_train,y_train)
y_pred5=model5.predict(X_test)

In [66]:
print("Accuracy:", accuracy_score(y_test, y_pred5))
print("Precision:", precision_score(y_test, y_pred5))
print("Recall:", recall_score(y_test, y_pred5))
print("F1 Score:", f1_score(y_test, y_pred5))
print(confusion_matrix(y_test, y_pred5))

Accuracy: 0.9894012237762237
Precision: 0.9937846836847947
Recall: 0.9848218213814343
F1 Score: 0.9892829521599824
[[4578   28]
 [  69 4477]]
